# 025 — Training: beta sweep on the remaining Laplacian NLL architectures

`024_training_beta_sweep.ipynb` swept the Laplace-NLL **beta exponent**
(`scripts.losses.laplace_nll_loss`) on `unet_nll` only — the pilot, chosen because
it has the healthiest sigma head of the current generation. This notebook extends
the **same grid** to the other three Laplacian NLL architectures so
`C4_beta_sweep_all.ipynb` can ask whether the beta effect measured on `unet_nll`
generalises — in particular for `resunet_nll`, the pixel-metric leader that is a
candidate for the Round 4 k-fold.

**What beta does** (recap, `024` has the full derivation). The loss is
`stop_gradient(b)^beta * (|y - mu| / b + log b)`. The detached weight does not move
the optimum for `b` (still `b* = |y - mu|`); it rescales each pixel's gradient to
`mu` as `1 / b^(1 - beta)`. `beta = 0` is the plain Laplace NLL (high-uncertainty
pixels downweighted most); `beta = 1` removes the weighting. Seitzer et al. (2022),
adapted from the Gaussian variance to the Laplace scale (`fixing.md` #10).

**Architectures**: `resunet_nll`, `attention_unet_nll`, `efficientnet_unet_nll`
(the three base Laplacian NLL heads `024` did not cover). `efficientnet_unet_nll_ft`
is left out — its fine-tuning was closed as a negative result (`fixing.md` §0), so a
beta sweep on it is not worth the compute.

**Grid**: `beta` in `{0.0, 0.25, 0.75}`, matching `024`. `beta = 0.5` is **not**
retrained — the existing `models/nll/<arch>/` checkpoint (from `021`/`022`) is that
point of the curve. `beta = 1.0` is left out as the degenerate end.

Everything else is held at the `021`/`022` recipe: same dataset, same split, same
seed, same optimizer, same callbacks. Only `beta` and the architecture move.

Runs are **skip-if-exists**: a `(beta, arch)` whose `best_model.keras` is already on
disk is not retrained, so `unet_nll`'s three `024` checkpoints are reused and a
re-run only trains what is missing.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.

In [ ]:
import json
import subprocess

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. The sweep grid

`scripts/train_single.py` rebuilds its own datasets from `settings` inside every
subprocess (deterministic from the seed), so — unlike the `020`-series notebooks —
there is nothing to construct here: the grid definition below is the whole setup.

`ARCHS` deliberately excludes `unet_nll` (already swept by `024`); add it back if you
want this notebook to reproduce those three runs too (they will be skipped if their
checkpoints exist).

In [ ]:
ARCHS = ["resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
LOSS_NAME = "laplace_nll"
BETAS = [0.0, 0.25, 0.75]  # 0.5 is the project default -> models/nll/<arch>/
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test

SWEEP_DIR = settings.MODELS_DIR / "beta_sweep"
SWEEP_LOG_DIR = settings.LOGS_DIR / "beta_sweep"

REFERENCE_BETA = settings.NLL_BETA           # 0.5
REFERENCE_DIR = settings.MODELS_DIR / "nll"  # trained by 021 / 022


def run_dir(root: Path, beta: float) -> Path:
    """``<root>/beta_<value>`` — one directory per sweep point, shared with 024."""
    return root / f"beta_{beta:.2f}"


print(f"Architectures: {ARCHS}")
print(f"Betas:         {BETAS}  (reference beta={REFERENCE_BETA}, not retrained)")
print(f"Runs to consider: {len(ARCHS) * len(BETAS)}  (existing checkpoints are skipped)")

## 2. Train one subprocess per `(architecture, beta)`

Same reasoning as `020_training.ipynb` §3 / `024` §3: training several Keras models
back-to-back in one process leaves GPU-side state behind that `clear_session()` does
not fully release on this hardware, so each run gets a fresh process
(`scripts/train_single.py --nll --nll-beta`).

Checkpoints land in `models/beta_sweep/beta_<value>/<arch>/best_model.keras` — the
exact layout `024` uses and `C3`/`C4` expect, with the architecture name kept intact
inside the per-beta directory so `load_model_nll(arch, model_dir=...)` resolves them
without special-casing. Each subprocess also writes `history.json` next to its
checkpoint.

In [ ]:
histories: dict[tuple[str, float], dict] = {}

for arch in ARCHS:
    for beta in BETAS:
        model_dir = run_dir(SWEEP_DIR, beta)
        ckpt = model_dir / arch / "best_model.keras"
        history_path = model_dir / arch / "history.json"

        if ckpt.exists():
            print(f"[skip] {arch} beta={beta} — checkpoint already at {ckpt}")
            if history_path.exists():
                histories[(arch, beta)] = json.loads(history_path.read_text())
            continue

        print(f"\n=== training {arch} (beta={beta}) ===")
        cmd = [
            sys.executable, "-m", "scripts.train_single",
            "--arch", arch,
            "--epochs", str(EPOCHS),
            "--model-dir", str(model_dir),
            "--log-dir", str(run_dir(SWEEP_LOG_DIR, beta)),
            "--nll",
            "--loss-name", LOSS_NAME,
            "--nll-beta", str(beta),
        ]
        subprocess.run(cmd, cwd=project_root, check=True)

        histories[(arch, beta)] = json.loads(history_path.read_text())
        best_val_loss = min(histories[(arch, beta)]["val_loss"])
        print(f"Best val_loss ({arch}, beta={beta}): {best_val_loss:.4f}")

print(f"\nHistories available for: {sorted(histories)}")

## 3. Pull in the `beta = 0.5` reference runs

`021`/`022` already trained every architecture at `beta = settings.NLL_BETA`; those
`history.json` files are read here so each curve has four points instead of three.
A missing reference is skipped, not an error.

In [ ]:
for arch in ARCHS:
    ref_path = REFERENCE_DIR / arch / "history.json"
    if ref_path.exists():
        histories[(arch, REFERENCE_BETA)] = json.loads(ref_path.read_text())
        print(f"reference loaded: {arch} beta={REFERENCE_BETA} <- {ref_path}")
    else:
        print(f"[skip] no reference run at {ref_path}")

swept_betas = sorted({b for _, b in histories})
print(f"\nBetas present: {swept_betas}")

## 4. Training curves

One figure per `(architecture, beta)`. A run that diverges or plateaus in the first
few epochs is a training failure, not a beta result — rerun it (delete its
checkpoint first) before reading anything into its metrics. `024`'s `unet_nll` runs
all early-stopped cleanly around epoch 22-26.

In [ ]:
for arch in ARCHS:
    for beta in sorted(swept_betas):
        if (arch, beta) not in histories:
            continue
        plot_training_curves(
            histories[(arch, beta)],
            title=f"Training history — {arch} ({LOSS_NAME}, beta={beta})",
        )
        plt.show()

## 5. Summary

Where every checkpoint landed, and its best `val_mae` (`MuMAEMetric`, the `mu`
channel only). Unlike `val_loss`, `val_mae` does **not** carry the beta weight, so it
is comparable across the sweep — a first, fidelity-only read on which beta trained
best for each architecture.

**This notebook decides nothing.** The sigma behaviour is the point of the sweep and
no training-time metric measures it — score these checkpoints in
`C4_beta_sweep_all.ipynb` (the §8 calibration block: `z_std` toward 1.0, `ence`
down, `error_sigma_spearman` up, coverage near nominal) before drawing any
conclusion.

In [ ]:
name_w = 22
header = "arch / beta".ljust(name_w) + "".join(f"beta={b}".rjust(20) for b in sorted(swept_betas))
print(header)
print("-" * len(header))

for arch in ARCHS:
    row = arch.ljust(name_w)
    for beta in sorted(swept_betas):
        root = REFERENCE_DIR if beta == REFERENCE_BETA else run_dir(SWEEP_DIR, beta)
        ckpt = root / arch / "best_model.keras"
        hist = histories.get((arch, beta))
        mae = f"{min(hist['val_mae']):.4f}" if hist and hist.get("val_mae") else "n/a"
        status = "ok" if ckpt.exists() else "MISSING"
        row += f"{mae} ({status})".rjust(20)
    print(row)

print(f"\nreference beta={REFERENCE_BETA} lives in {REFERENCE_DIR}/<arch>/ (from 021/022)")
print(f"swept betas live in {SWEEP_DIR}/beta_<value>/<arch>/")